# Step 2 — fact-aware case retrieval, as an ablation on Model A (thin notebook: it only calls scripts)
**Settings:** GPU T4 x2, Internet on, Secret `HF_TOKEN` ticked. Your HF account must have accepted the terms of
**google/medsiglip-448** (open its Hugging Face page once and click *Agree*), like MedGemma.
**Inputs:** the same datasets as Step 1 (Indiana, `rexgradient-160k-test-images`, `rexgradient-160k-train-subset`)
+ the outputs of your finished **Step 0** and **Step 1** notebooks.

Pipeline: MedSigLIP embeddings (train / val / test subset) -> CheXbert labels of train+val reports -> Probe ->
FAISS index over TRAIN images -> top-20 -> re-rank by label agreement with the Probe -> top-3 reports in the prompt
-> Model A with vs without retrieval, paired test, pre-registered keep/drop rule.

In [ ]:
!git clone -q --depth 1 https://github.com/Vinith-44/Radioreport /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
import os, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["RUN_NOTE"] = "step2 kaggle"

## 0. Restore Step 0 + Step 1 results (and a previous Step 2 session, to resume)

In [ ]:
STEP0 = None  # e.g. "/kaggle/input/step0-kaggle/project"
STEP1 = None  # e.g. "/kaggle/input/step1-kaggle/project"   (adapter + model-a predictions)
PREV2 = None  # e.g. "/kaggle/input/step2-kaggle/project"   (only to resume)
os.makedirs("/kaggle/working/project", exist_ok=True)
for src in [STEP0, STEP1, PREV2]:
    if src:
        !cp -rn {src}/outputs /kaggle/working/project/
        !cp -rn {src}/data /kaggle/working/project/ 2>/dev/null || true
!ls /kaggle/working/project/outputs/runs

## 1. Rebuild the parquet files (test subset must be identical to Step 0), run the tests

In [ ]:
ids_p = "/kaggle/working/project/data/step0_eval_ids.json"
before = json.load(open(ids_p)) if os.path.exists(ids_p) else None
!python scripts/step0_prepare_data.py --config configs/step2.yaml
assert before is None or before == json.load(open(ids_p)), "test eval ids changed - comparisons would be void"
print("eval ids unchanged" if before else "WARNING: attach STEP0 so the eval ids can be checked")
!python -m pytest tests -q

## 2. Smoke run (needs the Step 1 smoke run in the restored outputs)

In [ ]:
!python scripts/step0_prepare_data.py --config configs/step2.yaml --smoke
!python scripts/step2_embed.py --run-name r1 --smoke
!python scripts/step2_label_reports.py --run-name r1 --smoke
!python scripts/step2_train_probe.py --run-name r1 --smoke
!python scripts/step2_retrieve.py --run-name r1 --smoke
!python scripts/step2_generate.py --run-name r1 --smoke
!python scripts/step2_score.py --run-name r1 --smoke

## 3. Full run (every command is resumable; re-run after a timeout)

In [ ]:
!python scripts/step2_embed.py --run-name r1
!python scripts/step2_label_reports.py --run-name r1
!python scripts/step2_train_probe.py --run-name r1
!python scripts/step2_retrieve.py --run-name r1

In [ ]:
!python scripts/step2_generate.py --run-name r1
!python scripts/step2_score.py --run-name r1
!cat /kaggle/working/project/outputs/RESULTS.md

Optional extra ablation (similarity-only retrieval, no re-ranking) - tells whether the *fact-aware* part helps:

In [ ]:
# !python scripts/step2_retrieve.py --run-name r1 --mode similarity
# !python scripts/step2_generate.py --run-name r1 --mode similarity
# !python scripts/step2_score.py --run-name r1